### Note: Multiple GPUs are required to run code in this section

In [1]:
import torch 
from d2l import torch as d2l

In [2]:
devices = d2l.try_all_gpus()
def run(x):
    return [x.mm(x) for _ in range(50)]

x_gpu1 = torch.rand(size=(4000,4000), device=devices[0])
x_gpu2 = torch.rand(size=(4000,4000), device=devices[1])

IndexError: list index out of range

In [ ]:
# Warm up all devices
run(x_gpu1)
run(x_gpu2)
torch.cuda.synchronize(devices[0]) # Waits for all kernels in this device's stream to finish, synchronizing them
torch.cuda.synchronize(devices[1])

with d2l.Benchmark('GPU1 time'):
    run(x_gpu1)
    torch.cuda.synchronize(devices[0])
    
with d2l.Benchmark('GPU2 time'):
    run(x_gpu2)
    torch.cuda.synchronize(devices[1])

In [ ]:
# Alowing the system to parallelize
# System will automatically schedule computation on both GPU devices w/o need for extra code 
with d2l.Benchmark('GPU1 and GPU2'):
    run(x_gpu1)
    run(x_gpu2)
    torch.cuda.synchronize()

In [ ]:
def copy_to_cpu(x, non_blocking=False):
    return [y.to('cpu', non_blocking=non_blocking) for y in x]

with d2l.Benchmark('Run on GPU1'):
    y = run(x_gpu1)
    torch.cuda.synchronize()
    
with d2l.Benchmark('Copy to CPU'):
    y_cpu = copy_to_cpu(y)
    torch.cuda.synchronize()
    
with d2l.Benchmark('Run on GPU1 and copy to CPU'):
    y = run(x_gpu1)
    y_cpu - copy_to_cpu(y, True)
    torch.cuda.synchronize() # Total time required should be less than sum of parts

Note the dependency between computation and communication: `y[i]` must be computed before it can be copied to the CPU.

Fortunately, `y[i-1]` and `y[i]` can be computed at the same time.

# Exercises:

1. Eight operations were performed in the `run` function defined in this section. There are no dependencies between them. Design an experiment to see if the deep learning framework will automatically execute them in parallel.

1. When the workload of an individual operator is sufficiently small, paralellization can help, even on a single CPU or GPU. Design an experiment to verify this.

1. Design an experiment that uses parallel computaiton on CPUs, GPUs, and communication between both devices.

1. Use a debuger such as NVIDIA's Nsight to verify your code is efficient.

1. Designing computation tasss that incldue more complex data dependencies, and run experiments to see if you can obtain the correct results while improving performance.